In [5]:
from ultralytics import YOLO
import torch

In [8]:
print(torch.cuda.is_available())

True


In [ ]:
import os
from collections import Counter

train_label_dir = r"D:\LONGSON\2026_DUAN\DUAN2\detect_dump_truck\training\dataset\26_05_23_dump_detect_task\labels\train"
counts = Counter()
img_count = 0

for filename in os.listdir(train_label_dir):
  if filename.endswith(".txt"):
    img_count += 1
    with open(os.path.join(train_label_dir, filename), "r") as f:
      for line in f:
        parts = line.strip().split()
        if parts: # Nếu dòng không trống
          class_id = parts[0]
          counts[class_id] += 1
print(f"Tổng số ảnh có nhãn trong tập Train: {img_count}")
print(f"Tổng số instances (objects) trong tập Train:")
print(f" - truck_raised (Class 0) : {counts.get('0', 0)}")
print(f" - truck_lowered (Class 1): {counts.get('1', 0)}")


Tổng số ảnh có nhãn trong tập Train: 809
Tổng số instances (objects) trong tập Train:
 - truck_raised (Class 0) : 666
 - truck_lowered (Class 1): 276


Khi training với YOLO pre-trained model

In [ ]:
# 1. Load mô hình pretrained (bản Nano cho nhẹ và nhanh)
model = YOLO('training/model_loader/yolov8n.pt')

# 2. Huấn luyện
results = model.train(
    data='dataset/26_05_23_dump_detect_task/26-05-23-data.yaml',
    epochs=150,      # Số vòng lặp, PoC nên để 100-150
    imgsz=640,       # Kích thước ảnh đầu vào
    device=0,        # Sử dụng GPU (nếu có)
    batch=16,        # số lượng batch
    lr0=0.0005,
    workers=4,
    patience=40,
    project='26-05-23-truck-detect', # Tên thư mục lưu kết quả
    name='poc_v7.1',    # Tên đợt training này

    # augmentation nhẹ cho bài toán xe ben
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,

    degrees=5,
    translate=0.1,
    scale=0.3,

    fliplr=0.5,
    mosaic=0.5,
    close_mosaic=10,    # tắt mosaic 10 epoch cuối
    save=True           # Lưu checkpoint mỗi epoch
    
)

Khi fine-turning từ kết quả của lần runs trước đó

In [ ]:
# 1. Load model từ checkpoint trước
model = YOLO(
  'runs/detect/26-05-23-truck-detect/poc_v5/weights/last.pt'
)
# 2. Huấn luyện
results = model.train(
  data='dataset/26_05_23_dump_detect_task/26-05-23-data.yaml',
  epochs=200,      # => tăng lên 200 để model học ổn định hơn
  imgsz=640,       # Kích thước ảnh đầu vào
  device=0,        # Sử dụng GPU (nếu có)

  # Batch size nhỏ:
  # - phù hợp VRAM thấp - nhưng metrics dễ zig-zag hơn
  batch=16,

  workers=4,          # Số worker load data
  optimizer="AdamW",      # Ổn định hơn SGD cho fine-turning
  lr0=0.0005,          # learning rate thấp - fine-turn ổn định
  patience=40,         # early stopping sau 40 epoch nếu validation không cải thiện

  project='26-05-23-truck-detect',    # Tên thư mục lưu kết quả
  name='poc_v6_retrain',    # Tên đợt training này

  # augmentation nhẹ cho bài toán xe ben
  hsv_h=0.015,
  hsv_s=0.5,
  hsv_v=0.3,

  degrees=5,
  translate=0.1,
  scale=0.3,

  fliplr=0.5,
  mosaic=0.5,
  close_mosaic=10,    # tắt mosaic 10 epoch cuối

  pretrained=True,     # Quan trọng, cần có để train từ checkpoint trước đó
  save=True,           # Lưu checkpoint mỗi epoch
    
)